In [1]:
# Cell 1: Colab setup (run this first)
!pip -q install -U ultralytics pyyaml
from pathlib import Path
import shutil
import yaml
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

COLAB_ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive')
print('Colab root:', COLAB_ROOT)
print('Drive root:', DRIVE_ROOT)

Mounted at /content/drive
Colab root: /content
Drive root: /content/drive/MyDrive


In [2]:
# Cell 2: Configure dataset paths and create a Colab-friendly data YAML
PROJECT_DIR = COLAB_ROOT / 'emergency_yolo'
DATASET_DIR = PROJECT_DIR / 'data'
RUNS_DIR = PROJECT_DIR / 'runs'
EXPORT_DIR = DRIVE_ROOT / 'emergency_yolo_artifacts'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Preferred dataset sources in Drive
DRIVE_DATASET_DIR = DRIVE_ROOT / 'emergency_workbench' / 'data'
DRIVE_DATA_ZIP = DRIVE_ROOT / 'emergency_workbench' / 'data.zip'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Prepare dataset in Colab only if missing
if not DATASET_DIR.exists():
    if DRIVE_DATASET_DIR.exists():
        shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
        print('Dataset copied from Drive folder to Colab:', DATASET_DIR)
    elif DRIVE_DATA_ZIP.exists():
        print('Dataset folder not found. Unzipping from:', DRIVE_DATA_ZIP)
        tmp_extract = PROJECT_DIR / '_tmp_zip_extract'
        if tmp_extract.exists():
            shutil.rmtree(tmp_extract)
        tmp_extract.mkdir(parents=True, exist_ok=True)

        !unzip -q "{DRIVE_DATA_ZIP}" -d "{tmp_extract}"

        extracted_data_dir = tmp_extract / 'data'
        if extracted_data_dir.exists():
            shutil.move(str(extracted_data_dir), str(DATASET_DIR))
        else:
            DATASET_DIR.mkdir(parents=True, exist_ok=True)
            for item in tmp_extract.iterdir():
                shutil.move(str(item), str(DATASET_DIR / item.name))

        shutil.rmtree(tmp_extract)
        print('Dataset extracted to Colab:', DATASET_DIR)
    else:
        print('Drive dataset folder and zip not found.')
        print('Expected one of:')
        print(' -', DRIVE_DATASET_DIR)
        print(' -', DRIVE_DATA_ZIP)
else:
    print('Dataset already exists in Colab:', DATASET_DIR)

required = [DATASET_DIR / 'train/images',
            DATASET_DIR / 'valid/images',
            DATASET_DIR / 'test/images']
for p in required:
    print(p, 'exists:', p.exists())

# Build a corrected YAML regardless of the original relative paths
source_yaml = DATASET_DIR / 'data.yaml'
if not source_yaml.exists():
    raise FileNotFoundError(f'Missing dataset YAML: {source_yaml}')

with open(source_yaml, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

names = cfg.get('names', [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]

colab_yaml = PROJECT_DIR / 'data_colab.yaml'
fixed_cfg = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': int(cfg.get('nc', len(names))),
    'names': names
}

with open(colab_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(fixed_cfg, f, sort_keys=False, allow_unicode=False)

print('Generated:', colab_yaml)
print(open(colab_yaml, 'r', encoding='utf-8').read())

Dataset folder not found. Unzipping from: /content/drive/MyDrive/emergency_workbench/data.zip
Dataset extracted to Colab: /content/emergency_yolo/data
/content/emergency_yolo/data/train/images exists: True
/content/emergency_yolo/data/valid/images exists: True
/content/emergency_yolo/data/test/images exists: True
Generated: /content/emergency_yolo/data_colab.yaml
path: /content/emergency_yolo/data
train: train/images
val: valid/images
test: test/images
nc: 7
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'



In [5]:
# Cell 3: Create a smaller subset dataset for faster training
import random

TRAIN_LIMIT = 1000
VAL_LIMIT = 200
TEST_LIMIT = 200
SEED = 42

SUBSET_DATASET_DIR = PROJECT_DIR / 'data_subset'
if SUBSET_DATASET_DIR.exists():
    shutil.rmtree(SUBSET_DATASET_DIR)

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def get_image_label_pairs(split_name: str):
    image_dir = DATASET_DIR / split_name / 'images'
    label_dir = DATASET_DIR / split_name / 'labels'
    pairs = []

    if not image_dir.exists() or not label_dir.exists():
        return pairs

    for img_path in sorted(image_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in VALID_EXTS:
            continue
        label_path = label_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            pairs.append((img_path, label_path))

    return pairs


def sample_pairs(pairs, limit, seed):
    if len(pairs) <= limit:
        return pairs
    rng = random.Random(seed)
    return rng.sample(pairs, limit)


splits = [
    ('train', TRAIN_LIMIT, SEED + 1),
    ('valid', VAL_LIMIT, SEED + 2),
    ('test', TEST_LIMIT, SEED + 3),
]

summary = {}
for split_name, limit, split_seed in splits:
    all_pairs = get_image_label_pairs(split_name)
    chosen_pairs = sample_pairs(all_pairs, limit, split_seed)
    summary[split_name] = {'available': len(
        all_pairs), 'selected': len(chosen_pairs)}

    out_img_dir = SUBSET_DATASET_DIR / split_name / 'images'
    out_lbl_dir = SUBSET_DATASET_DIR / split_name / 'labels'
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_src, lbl_src in chosen_pairs:
        shutil.copy2(img_src, out_img_dir / img_src.name)
        shutil.copy2(lbl_src, out_lbl_dir / lbl_src.name)

print('Subset creation summary:', summary)

# Rebuild YAML to point to subset dataset
subset_yaml = PROJECT_DIR / 'data_colab_subset.yaml'
subset_cfg = {
    'path': str(SUBSET_DATASET_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': int(cfg.get('nc', len(names))),
    'names': names
}

with open(subset_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(subset_cfg, f, sort_keys=False, allow_unicode=False)

# Make training cell use subset YAML automatically
colab_yaml = subset_yaml

print('Subset dataset path:', SUBSET_DATASET_DIR)
print('Subset YAML:', subset_yaml)
print(open(subset_yaml, 'r', encoding='utf-8').read())

Subset creation summary: {'train': {'available': 5055, 'selected': 1000}, 'valid': {'available': 1245, 'selected': 200}, 'test': {'available': 466, 'selected': 200}}
Subset dataset path: /content/emergency_yolo/data_subset
Subset YAML: /content/emergency_yolo/data_colab_subset.yaml
path: /content/emergency_yolo/data_subset
train: train/images
val: valid/images
test: test/images
nc: 7
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'



In [6]:
# Cell 4: Fine-tune YOLOv8n and save best model to Drive
EPOCHS = 50
IMG_SIZE = 640
BATCH = 16
PATIENCE = 10  # Early stopping: stop if no val improvement for N epochs

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(colab_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name='yolov8n_emergency',
    exist_ok=True,
    pretrained=True,
    device=0  # Use GPU in Colab runtime
)

best_pt = RUNS_DIR / 'detect' / 'yolov8n_emergency' / 'weights' / 'best.pt'
if not best_pt.exists():
    # Fallback for newer run directory naming
    best_pt = Path(model.trainer.save_dir) / 'weights' / 'best.pt'

export_best = EXPORT_DIR / 'best_yolov8n_emergency.pt'
export_last = EXPORT_DIR / 'last_yolov8n_emergency.pt'
shutil.copy2(best_pt, export_best)

last_pt = best_pt.parent / 'last.pt'
if last_pt.exists():
    shutil.copy2(last_pt, export_last)

print('Best model:', best_pt)
print('Saved to Drive:', export_best)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/emergency_yolo/data_colab_subset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_emergency, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [7]:
# Cell 4: Inference on a single custom image from Drive
INFER_DIR = PROJECT_DIR / 'inference_outputs'
single_image = DRIVE_ROOT / 'emergency_workbench' / 'sample.jpg'  # update filename
if single_image.exists():
    _ = trained_model.predict(
        source=str(single_image),
        conf=0.25,
        imgsz=640,
        save=True,
        project=str(INFER_DIR),
        name='predict_single',
        exist_ok=True
    )
    print('Single image prediction saved in:', INFER_DIR / 'predict_single')
else:
    print('Put an image at:', single_image)

Put an image at: /content/drive/MyDrive/emergency_workbench/sample.jpg


In [9]:
# Cell 5: Inference on images and save predictions
INFER_DIR = PROJECT_DIR / 'inference_outputs'
sample_source = DATASET_DIR / 'test/images'
trained_model = YOLO(export_best)  # Load the best model for inference
pred_results = trained_model.predict(
    source=str(sample_source),
    conf=0.25,
    imgsz=640,
    save=True,
    project=str(INFER_DIR),
    name='predict_test',
    exist_ok=True
)

print('Predictions saved in:', INFER_DIR / 'predict_test')
print('Total predictions:', len(pred_results))


image 1/466 /content/emergency_yolo/data/test/images/0AXHYXBNS4UP_jpg.rf.13d244aec2ef4304a0ea76d7dc02d44c.jpg: 640x640 1 3, 8.0ms
image 2/466 /content/emergency_yolo/data/test/images/1000_F_212715308_QBQ1EMZkoFWu0yB5mNEVmWHyA7LFqwa6_jpg.rf.5442d266eacd89338598ef941de08df5.jpg: 640x640 2 1s, 7.5ms
image 3/466 /content/emergency_yolo/data/test/images/101_jpg.rf.322e90310781eab659c1b4d81748c30d.jpg: 640x640 1 5, 7.2ms
image 4/466 /content/emergency_yolo/data/test/images/101_jpg.rf.4ae683b6d4600b304777eca235b01f6a.jpg: 640x640 1 1, 1 3, 7.2ms
image 5/466 /content/emergency_yolo/data/test/images/101_jpg.rf.a5a7d49e716da7503a4d078fbf1b41bb.jpg: 640x640 1 1, 7.2ms
image 6/466 /content/emergency_yolo/data/test/images/102_jpg.rf.78104573f474ef308d96ec4be55adf69.jpg: 640x640 1 1, 7.2ms
image 7/466 /content/emergency_yolo/data/test/images/105_jpg.rf.157ebf4426322728aa5811eb0fd128e9.jpg: 640x640 1 1, 7.2ms
image 8/466 /content/emergency_yolo/data/test/images/105_jpg.rf.4c415b25433607a10d5b7bf8493

In [10]:
# Cell 4: Validate model on validation set
trained_model = YOLO(str(export_best))
metrics = trained_model.val(data=str(colab_yaml), split='val')
print(metrics)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1918.5±619.8 MB/s, size: 63.6 KB)
val: Scanning /content/emergency_yolo/data_subset/valid/labels.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 59.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.6it/s 5.0s0.2s
                   all        200        327      0.751      0.602        0.7      0.462
                     0         26         26      0.772      0.731      0.785      0.591
                     1         46        130      0.906      0.595      0.827      0.499
                     2         11         20      0.602      0.302       0.43      0.222
                     3         73         87      0.841      0.548       0.73      0.371
                     5    

In [11]:
# Cell 8: Save all generated artifacts to Google Drive (run last)
from datetime import datetime
import zipfile

STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
FINAL_EXPORT_DIR = DRIVE_ROOT / \
    'emergency_yolo_artifacts' / f'final_export_{STAMP}'
FINAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def copy_if_exists(src: Path, dst: Path):
    if src.exists():
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        print('Saved:', src, '->', dst)
    else:
        print('Skip (missing):', src)


# 1) Training outputs (check both possible run locations)
copy_if_exists(RUNS_DIR, FINAL_EXPORT_DIR / 'runs')
copy_if_exists(PROJECT_DIR / 'runs', FINAL_EXPORT_DIR / 'runs_project')

# 2) Inference outputs
copy_if_exists(PROJECT_DIR / 'inference_outputs',
               FINAL_EXPORT_DIR / 'inference_outputs')

# 3) Important model/config files
copy_if_exists(PROJECT_DIR / 'data_colab.yaml',
               FINAL_EXPORT_DIR / 'configs' / 'data_colab.yaml')
copy_if_exists(EXPORT_DIR / 'best_yolov8n_emergency.pt',
               FINAL_EXPORT_DIR / 'models' / 'best_yolov8n_emergency.pt')
copy_if_exists(EXPORT_DIR / 'last_yolov8n_emergency.pt',
               FINAL_EXPORT_DIR / 'models' / 'last_yolov8n_emergency.pt')

# 4) Also include the full PROJECT_DIR snapshot
copy_if_exists(PROJECT_DIR, FINAL_EXPORT_DIR / 'colab_project_snapshot')

# 5) Zip the final export folder for easy download/sharing
zip_path = FINAL_EXPORT_DIR.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in FINAL_EXPORT_DIR.rglob('*'):
        if fpath.is_file():
            zf.write(fpath, arcname=fpath.relative_to(FINAL_EXPORT_DIR.parent))

print('\nFinal export folder:', FINAL_EXPORT_DIR)
print('Final export zip:', zip_path)

Saved: /content/emergency_yolo/runs -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/runs
Saved: /content/emergency_yolo/runs -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/runs_project
Saved: /content/emergency_yolo/inference_outputs -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/inference_outputs
Saved: /content/emergency_yolo/data_colab.yaml -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/configs/data_colab.yaml
Saved: /content/drive/MyDrive/emergency_yolo_artifacts/best_yolov8n_emergency.pt -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/models/best_yolov8n_emergency.pt
Saved: /content/drive/MyDrive/emergency_yolo_artifacts/last_yolov8n_emergency.pt -> /content/drive/MyDrive/emergency_yolo_artifacts/final_export_20260404_181826/models/last_yolov8n_emergency.pt
Saved: /content/emergency_yolo -> /content/drive/MyDrive/emergen